In [11]:
import numpy as np
import numpy.testing as npt
from scipy import stats
from scipy.stats import t,ttest_ind
from scipy.stats import f
from scipy.stats import f_oneway
import statsmodels.api as sm
from statsmodels.regression._prediction import get_prediction
from statsmodels.stats.outliers_influence import OLSInfluence,MLEInfluence
from statsmodels.graphics.gofplots import qqplot_2samples,ProbPlot,qqplot
import pandas as pd
from patsy import dmatrices
from numpy.testing import assert_almost_equal, assert_allclose
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns

# some_file.py
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, r'C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\statemodelsStudy')
from olsRegressionAnalysis import dispAnalysisOfVariance, tableDispFormatt,getInvOfProductMat,getRegressionEqn,\
                                  dispReghressionAnalysis


In [12]:
# https://online.stat.psu.edu/stat462/node/165/ 
path  = r"C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\Notes\mlx\stat_analysis\CH-8-INDICATOR_VARIABLES\IndecatorVarPythonCode\Dataset\depression.txt"
df = pd.read_csv(path)
#print(df.columns)


### About Data Set:

 omparing the effectiveness of three treatments for severe depression.
 we denote the three treatments A, B, and C

So Models Eqn Become:

y =  β0 + β1age + β2A + β3B + β4age*A + β5age*B + ε

If patient receives treatments A So put A = 1 & B = 0

y = (β0 + β2) + (β1 + β4)age + ε

If patient receives treatments B So put A = 0 & B = 1

y = (β0 + β3) + (β1 + β5)age + ε

If patient receives treatments C So put A = 0 & B = 0

y =  β0 + β1age + ε


In [13]:
df = pd.get_dummies(df, columns=['TRT', ])
df.columns = ['y',  'age',  'A',  'B',  'C']
df.drop('C',axis='columns',inplace=True)

In [14]:
# You don't need to do this step
ser = pd.Series({True:1,False:0})
df['A'] = df['A'].map(ser)
df['B'] = df['B'].map(ser)

# Crossd Product
df['age_A'] = df['age'] * df['A']
df['age_B'] = df['age'] * df['B']

In [15]:
#print(df)

tableDispFormatt('estimating the full model')
y, X = dmatrices(
                 formula_like = ' y ~ age + A + B + age_A + age_B', 
                 data=df,
                 return_type='dataframe'
                 )
res = sm.OLS(y, X).fit()
getRegressionEqn(res)

=============================== estimating the full model ==================================
=============================== Regression eqn =============================================
6.211  + 1.033 age + 41.304 A + 22.707 B -0.703 age_A -0.51 age_B 


'6.211  + 1.033 age + 41.304 A + 22.707 B -0.703 age_A -0.51 age_B '

In [16]:
SSres_fm = res.ssr
MSres_fm = res.mse_model
SSr_fm   = res.ess
MSr_fm   = res.mse_resid
df_full_models = res.df_model # number of group
dfd_fm = res.df_resid


Full modela eqn

y = 6.211  + 1.033 age + 41.304 A + 22.707 B -0.703 age_A -0.51 age_B

Now Estimated regression function due to

A). If patient receives A, put A =1, B = 0 in model eqn

y = 6.211 + 1.033 age + 41.304*1 + 22.707 * 0 -0.703 age*1 -0.51 age * 0

y = 47.5 + 0.33age

B). If patient receives B,

TODO

C). If patient receives C

TODO

### Next Step check Hypothesis


Research question. For every age, 

is there a difference in the mean effectiveness 
for the three treatments?

Null Hypthesis: H0 : β2 = β3 = β4 = β5 = 0

Need to calculate

SSr(β2β3β4β5|β0β1) = SSr_fm - SSr_rm

                   = SSr(β0β1β2β3β4β5) - SSr(β0β1)


In [17]:
# get regression param redused models
tableDispFormatt('estimating the Redused model')
y, X = dmatrices(
                 formula_like = ' y ~ age', 
                 data=df,
                 return_type='dataframe'
                 )
res = sm.OLS(y, X).fit()
SSres_rm = res.ssr
MSres_rm = res.mse_model
SSr_rm   = res.ess
MSr_rm   = res.mse_resid

=============================== estimating the Redused model ===============================


In [18]:
# Calculate SSr due to beta(crossprod) & beta(Encode)
SSReffect = SSr_fm - SSr_rm 
MSresDueToFullModels = MSr_fm

In [19]:
# Step 4: Calculate F score
r = 4 # Calculating effect of two variable in rediused models
F = (SSReffect/r)/MSresDueToFullModels
# Calculate significance 95%
FSig = f.isf(q = 0.05, dfn = r,dfd = dfd_fm, loc=0, scale=1)
f_rm_pvalue = f.sf(F, dfn = r,dfd = dfd_fm)
tableDispFormatt('hypothesis mean effectiveness for the three treatments')
print('Fstae: ',F, ' FSig: ',FSig, ' p-val: ',f_rm_pvalue,' DFD ',dfd_fm)

=============================== hypothesis mean effectiveness for the three treatments =====
Fstae:  24.47951731055477  FSig:  2.6896275736914177  p-val:  4.457884916694127e-09  DFD  30.0



Research question. 

Does the effect of age on the treatment's effectiveness depend on treatment?

TODO


TODO

Another hypthesis test

If patient receives treatments A So put A = 1 & B = 0

y = (β0 + β2) + (β1 + β4)age + ε  --- (i)

If patient receives treatments B So put A = 0 & B = 1

y = (β0 + β3) + (β1 + β5)age + ε  --- (ii)

Check sloap of tretment A & treatment B are commons

NULL HYPOTHESIS H0 : β4 = β5 possible wrong hypothesis

                Hα : β4 ≠ β5

You can check this by linear hypthesis eqn
